In [2]:
import ee
import geemap
import os

ee.Initialize(project="albaniasat")
print("GEE connected!")

GEE connected!


In [3]:
albania = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
            .filter(ee.Filter.eq("country_na", "Albania"))

print("Albania boundary loaded")

Albania boundary loaded


In [4]:
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(albania.geometry()) \
        .filterDate("2021-01-01", "2021-12-31") \
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10)) \
        .select(["B4", "B3", "B2"])

print(f"Images found: {s2.size().getInfo()}")

Images found: 448


In [5]:
# Create one clean composite image from all 448 scenes
composite = s2.median()

# Add NIR band
composite = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(albania.geometry())
                .filterDate("2021-06-01", "2021-09-30")
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
                .select(["B4", "B3", "B2", "B8"])
                .median())

print("Composite image ready!")
info = composite.getInfo()
print("Bands:", [b["id"] for b in info["bands"]])

Composite image ready!
Bands: ['B4', 'B3', 'B2', 'B8']


In [6]:
import math

# Get Albania geometry
albania_geom = albania.geometry()
bounds = albania_geom.bounds().getInfo()["coordinates"][0]

# Albania bounds
min_lon = 19.3
max_lon = 21.1
min_lat = 39.6
max_lat = 42.7

# Each patch = 64 pixels * 10m = 640m ≈ 0.006 degrees
patch_size_deg = 0.006

# Count how many patches we'll have
cols = math.ceil((max_lon - min_lon) / patch_size_deg)
rows = math.ceil((max_lat - min_lat) / patch_size_deg)
print(f"Grid size: {cols} cols x {rows} rows = {cols * rows} total patches")

Grid size: 301 cols x 517 rows = 155617 total patches


155617 patches is way too many to export. We should aim for around 27000, 2000-3000 per class, across 10 classes.

Step 1  →  Load CORINE land cover map

Step 2  →  Define our classes (e.g. Forest, Farmland, Urban...)

Step 3  →  For each class, randomly sample N patch locations

Step 4  →  Export only those patches (labeled and ready)